In [13]:
!pip install openpyxl



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import os

# Path to your folder containing the 12 CSV files
data_folder = "C:/Users/prati/football_transfer_market_analytics/data" 
output_excel = "transfermarkt_data_dictionary.xlsx"

all_columns_data = []

# Loop through every CSV file and scan columns
for file_name in os.listdir(data_folder):
    if file_name.endswith('.csv'):
        file_path = os.path.join(data_folder, file_name)
        table_name = file_name.replace('.csv', '') 
        
        print(f"Processing table patterns for: {file_name}")
        df = pd.read_csv(file_path, nrows=2)
        
        for col in df.columns:
            raw_data_type = str(df[col].dtype)
            
            # --- Default Baseline Values ---
            guessed_type = "VARCHAR(255)"
            key_type = ""
            cleaning_rule = "Standard data type validation check."
            
            # --- 1. Base Type Guessing ---
            if pd.api.types.is_integer_dtype(df[col]):
                guessed_type = "INT"
            elif pd.api.types.is_float_dtype(df[col]):
                guessed_type = "DECIMAL(10,2)"
            elif "date" in col.lower():
                guessed_type = "DATE"
                cleaning_rule = "Standardize string format to YYYY-MM-DD; default missing or malformed to NULL."

            # --- 2. Automated Key & Cleaning Rule Logic ---
            # Pattern A: Primary Keys (e.g., game_id in games table, club_id in clubs table)
            if col == f"{table_name[:-1]}_id" or col == f"{table_name}_id" or (table_name == "competitions" and col == "competition_id"):
                key_type = "PK"
                guessed_type = "VARCHAR(255)" if table_name == "competitions" else "INT"
                cleaning_rule = "Enforce uniqueness constraint. Verify no missing or NULL values exist."
            
            # Pattern B: Foreign Keys (Any ID column that isn't the primary key)
            elif col.endswith("_id"):
                key_type = "FK"
                cleaning_rule = "Validate referential integrity against parent table. Treat missing rows as NULL."
                
                # Critical Catch: Fixes the Python "Float-ID Anomaly" where missing IDs force columns to float64
                if "float" in raw_data_type:
                    guessed_type = "INT"
                    cleaning_rule = "CRITICAL: Fix Float anomaly. Remove decimals (e.g., 14.0 -> 14) and default NaN to NULL."

            # Pattern C: Financial & Market Values
            elif "value" in col.lower() or "eur" in col.lower():
                if "float" in raw_data_type or "int" in raw_data_type:
                    guessed_type = "BIGINT" if "total" in col.lower() else "INT"
                cleaning_rule = "Ensure formatting strips out symbols (€). Impute missing values with 0 or median baseline."

            # --- 3. Append Row Data ---
            all_columns_data.append({
                "Source_Table": table_name,
                "Column_Name": col,
                "Raw_Data_Type": raw_data_type,
                "Target_SQL_Type": guessed_type,
                "Key_Type": key_type,         
                "Cleaning_Rule": cleaning_rule      
            })

# Save to Excel with formatting
blueprint_df = pd.DataFrame(all_columns_data)

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    blueprint_df.to_excel(writer, sheet_name='Data Dictionary', index=False)
    worksheet = writer.sheets['Data Dictionary']
    
    # Auto-fit columns dynamically
    for col in worksheet.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = col[0].column_letter
        worksheet.column_dimensions[col_letter].width = max(max_len + 3, 10)

print("\nSuccess! Data dictionary generated with automated keys and cleaning rules applied.")

Processing table patterns for: appearances.csv
Processing table patterns for: clubs.csv
Processing table patterns for: club_games.csv
Processing table patterns for: competitions.csv
Processing table patterns for: countries.csv
Processing table patterns for: games.csv
Processing table patterns for: game_events.csv
Processing table patterns for: game_lineups.csv
Processing table patterns for: national_teams.csv
Processing table patterns for: players.csv
Processing table patterns for: player_valuations.csv
Processing table patterns for: transfers.csv

Success! Data dictionary generated with automated keys and cleaning rules applied.
